## Download data from sports-reference/cbb

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment
from io import StringIO
from pathlib import Path

url = "https://www.sports-reference.com/cbb/schools/colorado/men/2027.html"

table_names = [
    "Roster",
    "Per Game Team and Opponents Stats",
    "Total Team and Opponents Stats",
    "Per Game",
    "Totals",
    "Per 40 Minutes",
    "Per 100 Possessions",
    "Advanced"
]

output_dir = Path("colorado_basketball_2027")
output_dir.mkdir(exist_ok=True)

# Download webpage
response = requests.get(url, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Find regular tables
tables = soup.find_all("table")

# Find tables inside HTML comments
for comment in soup.find_all(string=lambda x: isinstance(x, Comment)):
    if "<table" in comment:
        hidden_soup = BeautifulSoup(str(comment), "html.parser")
        tables.extend(hidden_soup.find_all("table"))

# Remove duplicate tables by ID
unique_tables = {}
for i, table in enumerate(tables):
    table_id = table.get("id", f"table_{i}")
    unique_tables[table_id] = table

print(f"Found {len(unique_tables)} tables")

# Print table IDs to verify their order
for i, table_id in enumerate(unique_tables):
    print(i, table_id)

# Only assign names if exactly 8 tables were found
if len(unique_tables) != 8:
    raise ValueError(
        f"Expected 8 tables, found {len(unique_tables)}. "
        "Check the table IDs before saving."
    )

# Save each table as CSV
for name, table in zip(table_names, unique_tables.values()):
    df = pd.read_html(StringIO(str(table)))[0]

    # Flatten multi-level column names
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join(str(x) for x in col if "Unnamed" not in str(x))
            for col in df.columns
        ]

    df.to_csv(output_dir / f"{name}.csv", index=False)
    print(f"Saved: {name}.csv")

Found 8 tables
0 roster
1 season-total_per_game
2 season-total_totals
3 players_per_game
4 players_totals
5 players_per_min
6 players_per_poss
7 players_advanced
Saved: Roster.csv
Saved: Per Game Team and Opponents Stats.csv
Saved: Total Team and Opponents Stats.csv
Saved: Per Game.csv
Saved: Totals.csv
Saved: Per 40 Minutes.csv
Saved: Per 100 Possessions.csv
Saved: Advanced.csv
